In [2]:
from pyspark.sql.functions import col, lit, current_timestamp, to_date, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, DecimalType

# Load both cleaned tables
df_excel = spark.sql("SELECT * FROM HR_Analytics_Lakehouse.STG.employee_excel_data_cleaned")
df_txt = spark.sql("SELECT * FROM HR_Analytics_Lakehouse.STG.employee_txt_data_cleaned")

print(f"Excel rows: {df_excel.count()}")
print(f"TXT rows: {df_txt.count()}")

# Standardize LoadTimestamp to current timestamp
from pyspark.sql.functions import current_timestamp

df_excel = df_excel.withColumn('LoadTimestamp', current_timestamp())
df_txt = df_txt.withColumn('LoadTimestamp', current_timestamp())

# Update SourceLocation
df_excel = df_excel.withColumn('SourceLocation', lit('STG.employee_excel_data_cleaned'))
df_txt = df_txt.withColumn('SourceLocation', lit('STG.employee_txt_data_cleaned'))

# Find exact duplicate rows (all columns match) between the two tables
# Use exceptAll to find rows in TXT that are NOT exact duplicates of any Excel row
txt_unique = df_txt.exceptAll(df_excel)
print(f"\nTXT rows that are exact duplicates of Excel rows: {df_txt.count() - txt_unique.count()}")

# Combine: all Excel + only unique TXT rows
df_combined = df_excel.unionByName(txt_unique)

print(f"\nCombined rows: {df_combined.count()}")
print(f"Expected: {df_excel.count() + txt_unique.count()}")

# Verify
print(f"\nDuplicate EmployeeIds after merge: {df_combined.count() - df_combined.select('EmployeeId').distinct().count()}")

df_combined.printSchema()
df_combined.show(5, truncate=False)

StatementMeta(, 3ebcdea8-90b5-46e4-9341-3b6a16c71c89, 4, Finished, Available, Finished, False)

Excel rows: 1000
TXT rows: 1000

TXT rows that are exact duplicates of Excel rows: 0

Combined rows: 2000
Expected: 2000

Duplicate EmployeeIds after merge: 999
root
 |-- EmployeeId: integer (nullable = true)
 |-- FullName: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- DepartmentName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- MonthlySalary: decimal(10,3) (nullable = true)
 |-- Gender: string (nullable = true)
 |-- JobTitle: string (nullable = true)
 |-- EmployeeStatus: string (nullable = true)
 |-- EducationLevel: string (nullable = true)
 |-- LoadTimestamp: timestamp (nullable = false)
 |-- SourceLocation: string (nullable = false)
 |-- FileType: string (nullable = true)
 |-- WasError: string (nullable = true)

+----------+-------------+---------+--------+----------+--------------+------------+-------------+------+------------------------+--------------+--

In [3]:
# Write to STG schema as a managed table
df_combined.write.mode("overwrite").saveAsTable("STG.employee_combined_cleaned_data")

print(f"Saved {df_combined.count()} rows to STG.employee_combined_cleaned_data")

StatementMeta(, 3ebcdea8-90b5-46e4-9341-3b6a16c71c89, 5, Finished, Available, Finished, False)

Saved 2000 rows to STG.employee_combined_cleaned_data
